# ICA — unmixing independent signals (FastICA)

> Tutorial pair for [`ica.py`](ica.py).

## 1. Intuition
Several microphones each record a different *mixture* of the same voices (the cocktail
party). ICA recovers the individual voices knowing **only** the mixtures — no info about
who sat where. The key insight: real signals are **non-Gaussian**, and any mixture of
them looks **more Gaussian** than the originals. So "un-mixing" = "making the outputs as
non-Gaussian (and independent) as possible".

## 2. Concept (the slide)
- Model: observed $\mathbf x = A\mathbf s$, with unknown mixing matrix $A$ and
  statistically **independent** sources $\mathbf s$. Find $W\approx A^{-1}$ so
  $\hat{\mathbf s}=W\mathbf x$ are independent.
- **Why non-Gaussianity?** Central Limit Theorem: sums of independents drift toward
  Gaussian. Maximizing non-Gaussianity of projections recovers the sources.
- Measure non-Gaussianity by **negentropy**, approximated with nonlinear contrasts
  (logcosh, exp, kurtosis).
- **Whiten** first (decorrelate + unit variance) → the unmixing reduces to finding an
  **orthogonal** matrix, solved by a **fixed-point** iteration (FastICA).
- **Ambiguities:** sign and permutation of the recovered sources are arbitrary.

## 3. Math derivation

**Model & goal.** $\mathbf x=A\mathbf s$; we seek $\mathbf y=W\mathbf x$ with components
as independent as possible. Independence $\Rightarrow$ each $y_i$ should be maximally
non-Gaussian (CLT argument).

**Negentropy.** For unit-variance $y$, negentropy
$J(y)=H(y_\text{gauss})-H(y)\ge0$ is zero **iff** $y$ is Gaussian — a principled
non-Gaussianity measure. It's hard to compute, so approximate with a smooth contrast $G$:

$$J(y)\;\approx\;\big[\,\mathbb E\{G(y)\}-\mathbb E\{G(\nu)\}\,\big]^2,\qquad \nu\sim\mathcal N(0,1).$$

Common choices ($g=G'$): $\,G=\frac1a\log\cosh(a u)\Rightarrow g=\tanh(au)$;
$\,G=-e^{-u^2/2}\Rightarrow g=u\,e^{-u^2/2}$; kurtosis $G=u^4/4\Rightarrow g=u^3$.

**Whitening (mandatory pre-step).** Center, then transform $\mathbf x_w=K\mathbf x$ so
$\mathbb E\{\mathbf x_w\mathbf x_w^\top\}=I$. From the covariance eigendecomposition
$C=E D E^\top$, take $K=D^{-1/2}E^\top$. After whitening the unmixing matrix is
**orthogonal**, shrinking the search space.

**FastICA fixed point.** Maximize $\mathbb E\{G(\mathbf w^\top\mathbf x_w)\}$ subject to
$\lVert\mathbf w\rVert=1$. The Lagrange/Newton step yields the celebrated update

$$\boxed{\;\mathbf w^+=\mathbb E\{\mathbf x_w\,g(\mathbf w^\top\mathbf x_w)\}-\mathbb E\{g'(\mathbf w^\top\mathbf x_w)\}\,\mathbf w,\qquad \mathbf w\leftarrow\mathbf w^+/\lVert\mathbf w^+\rVert\;}$$

It converges **cubically** for the kurtosis contrast — far faster than gradient ascent
(hence *Fast*ICA).

**Getting several components.**
- *Deflation:* find $\mathbf w$'s one at a time, **Gram–Schmidt**-orthogonalizing each
  new one against those already found:
  $\mathbf w\leftarrow\mathbf w-\sum_{j<i}(\mathbf w^\top\mathbf w_j)\mathbf w_j$.
- *Symmetric:* update all rows together, then **symmetric decorrelation**
  $W\leftarrow(WW^\top)^{-1/2}W$ (no component is privileged).

**Recover sources.** $\hat{\mathbf s}=W K(\mathbf x-\bar{\mathbf x})$. Note ICA fixes
neither the **sign** nor the **order** nor the **scale** of the sources.

## 4. NumPy implementation (whitening + fixed-point, deflation & symmetric)

In [ ]:
# ===== actual implementation from ica.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _logcosh(u, alpha=1.0):
    # G(u) = (1/a) log cosh(a u);  g = tanh(a u);  g' = a (1 - tanh²)
    gu = np.tanh(alpha * u)
    g_prime = alpha * (1.0 - gu ** 2)
    return gu, g_prime

def _exp(u):
    # G(u) = -exp(-u²/2);  g = u exp(-u²/2);  g' = (1 - u²) exp(-u²/2)
    e = np.exp(-(u ** 2) / 2.0)
    return u * e, (1.0 - u ** 2) * e

def _cube(u):
    # kurtosis-based contrast G(u)=u⁴/4;  g = u³;  g' = 3u²
    return u ** 3, 3.0 * u ** 2

_CONTRASTS = {"logcosh": _logcosh, "exp": _exp, "cube": _cube}

class FastICANumPy:
    r"""
    FastICA.  Steps:

    1. **Center** X (zero mean per feature).
    2. **Whiten**: find K so that  X_w = K X  has identity covariance. Using the
       eigendecomposition of the covariance  C = E D Eᵀ,  K = D^{-1/2} Eᵀ.
       After whitening the unmixing reduces to finding an *orthogonal* W.
    3. **Fixed-point iteration** for each unit-norm row w:
           w⁺ = E[ x_w g(wᵀx_w) ] − E[ g'(wᵀx_w) ] w,
           w  = w⁺ / ‖w⁺‖.
       This is a Newton step maximizing the negentropy approximation
       J(wᵀx) ≈ [E{G(wᵀx)} − E{G(ν)}]²  (ν standard Gaussian).
    4. Deflation: orthogonalize each new w against the already-found ones
       (Gram–Schmidt) so components stay distinct.
    """

    def __init__(self, n_components=None, fun="logcosh", max_iter=200,
                 tol=1e-5, algorithm="deflation", seed=SEED):
        self.n_components = n_components
        self.fun = fun
        self.max_iter = max_iter
        self.tol = tol
        self.algorithm = algorithm
        self.seed = seed

    def _whiten(self, X):
        # X is (features, samples). Center across samples.
        self.mean_ = X.mean(1, keepdims=True)
        Xc = X - self.mean_
        cov = (Xc @ Xc.T) / Xc.shape[1]            # (d, d) covariance
        d, E = np.linalg.eigh(cov)                  # eigvals ascending
        d = np.maximum(d, 1e-12)
        # whitening matrix K = D^{-1/2} Eᵀ so that cov(K Xc) = I
        K = np.diag(1.0 / np.sqrt(d)) @ E.T
        Xw = K @ Xc                                 # whitened: identity covariance
        return Xw, K

    def _g(self, u):
        return _CONTRASTS[self.fun](u)

    def fit(self, X):
        # Accept X as (samples, features); work internally as (features, samples).
        X = np.asarray(X, float).T
        n_features = X.shape[0]
        k = self.n_components or n_features
        rng = np.random.default_rng(self.seed)

        Xw, K = self._whiten(X)
        m = Xw.shape[1]                             # number of samples

        if self.algorithm == "deflation":
            W = np.zeros((k, Xw.shape[0]))
            for i in range(k):
                w = rng.standard_normal(Xw.shape[0])
                w /= np.linalg.norm(w)
                for _ in range(self.max_iter):
                    wx = w @ Xw                     # projection (1, m)
                    g, gp = self._g(wx)
                    # fixed-point update (the FastICA Newton step)
                    w_new = (Xw * g).mean(1) - gp.mean() * w
                    # Gram–Schmidt: remove components already found (deflation)
                    w_new -= W[:i].T @ (W[:i] @ w_new)
                    w_new /= np.linalg.norm(w_new) + 1e-12
                    if np.abs(np.abs(w_new @ w) - 1.0) < self.tol:
                        w = w_new
                        break
                    w = w_new
                W[i] = w
        else:  # symmetric: update all rows, then symmetric decorrelation
            W = rng.standard_normal((k, Xw.shape[0]))
            W = self._sym_decorrelate(W)
            for _ in range(self.max_iter):
                WX = W @ Xw                         # (k, m)
                g, gp = self._g(WX)
                W_new = (g @ Xw.T) / m - (gp.mean(1)[:, None] * W)
                W_new = self._sym_decorrelate(W_new)
                # convergence: largest change in alignment
                lim = np.max(np.abs(np.abs(np.einsum("ij,ij->i", W_new, W)) - 1.0))
                W = W_new
                if lim < self.tol:
                    break

        self.components_ = W                        # unmixing in whitened space
        self.whitening_ = K
        self.unmixing_ = W @ K                      # full unmixing: S = (W K)(X - mean)
        self.mixing_ = np.linalg.pinv(self.unmixing_)
        return self

    @staticmethod
    def _sym_decorrelate(W):
        # W <- (W Wᵀ)^{-1/2} W  (makes rows orthonormal without preferring order)
        s, U = np.linalg.eigh(W @ W.T)
        s = np.maximum(s, 1e-12)
        return (U * (1.0 / np.sqrt(s))) @ U.T @ W

    def transform(self, X):
        X = np.asarray(X, float).T
        S = self.unmixing_ @ (X - self.mean_)
        return S.T                                  # (samples, components)

    def fit_transform(self, X):
        return self.fit(X).transform(X)

## 5. PyTorch implementation (symmetric FastICA with torch.linalg)

In [ ]:
# ===== actual implementation from ica.py =====
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def fastica_torch(X, n_components=None, fun="logcosh", max_iter=200,
                  tol=1e-5, seed=SEED):
    r"""
    FastICA (symmetric) with torch tensors. Same whitening + fixed-point math;
    uses torch.linalg for the eigendecompositions. The fixed-point map is not a
    gradient step, so we run it explicitly (no autograd needed).
    """
    dev = get_device()
    Xt = torch.as_tensor(np.asarray(X), dtype=torch.float64, device=dev).T
    d, m = Xt.shape
    k = n_components or d

    mean = Xt.mean(1, keepdim=True)
    Xc = Xt - mean
    cov = (Xc @ Xc.T) / m
    evals, E = torch.linalg.eigh(cov)
    evals = torch.clamp(evals, min=1e-12)
    K = torch.diag(1.0 / torch.sqrt(evals)) @ E.T
    Xw = K @ Xc

    def g_fn(u):
        if fun == "logcosh":
            gu = torch.tanh(u)
            return gu, (1.0 - gu ** 2)
        if fun == "exp":
            e = torch.exp(-(u ** 2) / 2)
            return u * e, (1.0 - u ** 2) * e
        return u ** 3, 3.0 * u ** 2                 # cube / kurtosis

    def sym_decorrelate(W):
        s, U = torch.linalg.eigh(W @ W.T)
        s = torch.clamp(s, min=1e-12)
        return (U * (1.0 / torch.sqrt(s))) @ U.T @ W

    gen = torch.Generator(device="cpu").manual_seed(seed)
    W = torch.randn(k, d, generator=gen).to(torch.float64).to(dev)
    W = sym_decorrelate(W)
    for _ in range(max_iter):
        WX = W @ Xw
        g, gp = g_fn(WX)
        W_new = (g @ Xw.T) / m - gp.mean(1)[:, None] * W
        W_new = sym_decorrelate(W_new)
        lim = (torch.abs(torch.einsum("ij,ij->i", W_new, W)) - 1.0).abs().max()
        W = W_new
        if lim < tol:
            break

    unmix = W @ K
    S = unmix @ Xc
    return S.T.cpu().numpy(), unmix.cpu().numpy()

def demo():
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    # Build 3 independent, non-Gaussian source signals.
    n = 2000
    t = np.linspace(0, 8, n)
    s1 = np.sin(2 * t)                              # sinusoid
    s2 = np.sign(np.sin(3 * t))                     # square wave
    rng = np.random.default_rng(SEED)
    s3 = rng.uniform(-1, 1, n)                      # uniform noise (saw-like)
    S = np.c_[s1, s2, s3]
    S /= S.std(0)                                   # unit variance per source

    # Mix them with a random mixing matrix A: X = S Aᵀ.
    A = np.array([[1.0, 1.0, 1.0],
                  [0.5, 2.0, 1.0],
                  [1.5, 1.0, 2.0]])
    X = S @ A.T

    # Recover sources with FastICA (deflation).
    ica = FastICANumPy(n_components=3, fun="logcosh", algorithm="deflation").fit(X)
    S_hat = ica.transform(X)

    # ICA has sign + permutation ambiguity; match recovered to true sources by
    # absolute correlation and report the best alignment.
    def best_corr(S_true, S_est):
        C = np.abs(np.corrcoef(S_true.T, S_est.T)[:3, 3:])   # 3x3 |corr|
        # greedily match each true source to its best estimate
        used, total = set(), 0.0
        for _ in range(3):
            i, j = np.unravel_index(np.argmax(C), C.shape)
            total += C[i, j]
            C[i, :] = -1
            C[:, j] = -1
        return total / 3

    print(f"deflation  mean |corr| recovered vs true: {best_corr(S, S_hat):.3f}")

    # Symmetric algorithm and a different contrast function.
    S_sym = FastICANumPy(3, fun="exp", algorithm="symmetric").fit_transform(X)
    print(f"symmetric  mean |corr| recovered vs true: {best_corr(S, S_sym):.3f}")

    # PCA cannot separate them (uncorrelated ≠ independent).
    Xc = X - X.mean(0)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    S_pca = Xc @ Vt.T
    print(f"PCA        mean |corr| recovered vs true: {best_corr(S, S_pca):.3f}"
          f"  (worse: PCA only decorrelates)")

    # Torch FastICA agrees.
    S_t, _ = fastica_torch(X, n_components=3, fun="logcosh")
    print(f"torch ICA  mean |corr| recovered vs true: {best_corr(S, S_t):.3f}")

    # Sanity: recovered sources are nearly uncorrelated with each other.
    off = np.abs(np.corrcoef(S_hat.T) - np.eye(3)).max()
    print(f"max off-diagonal |corr| among recovered: {off:.3f}")

## 6. Train / run — blind source separation vs PCA

In [ ]:
demo()

## 7. Visualization — sources, mixtures, and recovered signals

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import ica as M

np.random.seed(0)
n = 2000; t = np.linspace(0, 8, n)
s1 = np.sin(2 * t); s2 = np.sign(np.sin(3 * t))
s3 = np.random.default_rng(0).uniform(-1, 1, n)
S = np.c_[s1, s2, s3]; S /= S.std(0)
A = np.array([[1.0, 1.0, 1.0], [0.5, 2.0, 1.0], [1.5, 1.0, 2.0]])
X = S @ A.T
S_hat = M.FastICANumPy(n_components=3, fun="logcosh").fit_transform(X)

fig, ax = plt.subplots(3, 3, figsize=(13, 6), sharex=True)
for r, (data, title) in enumerate([(S, "sources"), (X, "mixtures"), (S_hat, "ICA recovered")]):
    for c in range(3):
        ax[r, c].plot(t[:400], data[:400, c], lw=0.8)
        if c == 0:
            ax[r, c].set_ylabel(title)
ax[0, 0].set_title("signal 1"); ax[0, 1].set_title("signal 2"); ax[0, 2].set_title("signal 3")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **Whitening is mandatory** and turns ICA into a search over orthogonal matrices.
- ICA needs **non-Gaussian** sources — it **cannot** separate two Gaussians (their
  mixture is rotationally symmetric, so the directions are unidentifiable).
- **PCA ≠ ICA:** PCA only *decorrelates* (2nd-order); ICA enforces full statistical
  *independence* (higher-order) — the demo shows PCA failing to unmix.
- Outputs have **arbitrary sign, scale, and order** — match to ground truth by
  correlation when evaluating.
- Choose the contrast for robustness: **logcosh** is a good general default; pure
  **kurtosis** ($u^3$) is fast but sensitive to outliers.